# Tarea 2: Fundamentos de Python
## Ciencia de Datos Ambientales - UTEC

**Nombre:** Katia Andrea Galindo Ponce  
**Puntaje total:** 20 puntos

**Instrucciones:**
- Completa todos los problemas en este notebook
- Escribe tu codigo en las celdas proporcionadas
- Ejecuta todas las celdas antes de entregar
- Sube el archivo `.ipynb` completado al modulo correspondiente en Canvas

**Integridad academica:** Tarea individual. Puedes consultar materiales del curso y documentacion de Python, pero todo el codigo debe ser tuyo.

---

## Problema 1: Procesador de Nombres de Archivos Landsat (10 puntos)

Trabajas con imagenes satelitales Landsat del Peru. Los nombres siguen el formato:

```
LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF
```
Componentes: `{sensor}_{nivel}_{path_row}_{fecha}_{coleccion}_{tier}_SR_{banda}.TIF`

> Los paths 003-009, filas 062-071 cubren el territorio peruano (Madre de Dios, Loreto, Lima, Cusco).

### Tus tareas:

**Parte A (4 pts):** Funcion `procesar_nombre_landsat(nombre_archivo)` que devuelva un diccionario con:
- `sensor` (ej. "LC08"), `path` (ej. "008"), `row` (ej. "067")
- `fecha` formateada como "AAAA-MM-DD"
- `banda` (ej. "B4")

**Parte B (3 pts):** Funcion `clasificar_banda(banda)` que devuelva el nombre segun la tabla:

| Banda | Nombre |
|-------|--------|
| B1 | Aerosol costero | B2 | Azul | B3 | Verde | B4 | Rojo |
| B5 | Infrarrojo cercano (NIR) | B6 | SWIR1 | B7 | SWIR2 |

Si no esta en la tabla, devuelve "Desconocida".

**Parte C (3 pts):** Procesa la lista de archivos: parsea, imprime resumen (fecha/path/row/banda) y cuenta cuantas fechas unicas hay.

In [12]:
archivos = [
    "LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF",
    "LC08_L2SP_008067_20240615_02_T1_SR_B5.TIF",
    "LC08_L2SP_008067_20240701_02_T1_SR_B3.TIF",
    "LC08_L2SP_009067_20240615_02_T1_SR_B4.TIF",
    "LC09_L2SP_008067_20240708_02_T1_SR_B6.TIF",
    "LC08_L2SP_008067_20240701_02_T1_SR_B4.TIF",
]

# Parte A: funcion procesar_nombre_landsat

def procesar_nombre_landsat(nombre_archivo):

  division = nombre_archivo.split("_")

  sensor = division[0]

  # division de path_row
  path_row = division[2]
  path = path_row[:3]
  row = path_row[3:]

  # modificacion de fecha AAAA-MM-DD
  fecha_completa = division[3]
  fecha_modificada = f"{fecha_completa[:4]}-{fecha_completa[4:6]}-{fecha_completa[6:]}"

  # division de banda
  banda_completa = division[7]
  banda = banda_completa.split(".")[0]

  return{
        "sensor": sensor,
        "path": path,
        "row": row,
        "fecha": fecha_modificada,
        "banda": banda
    }


# Parte B: funcion clasificar_banda

def clasificar_banda(banda):
  if banda == "B1":
    return "Aerosol costero"
  elif banda == "B2":
    return "Azul"
  elif banda == "B3":
    return "Verde"
  elif banda == "B4":
    return "Rojo"
  elif banda == "B5":
    return "Infrarrojo cercano (NIR)"
  elif banda == "B6":
    return "SWIR1"
  elif banda == "B7":
    return "SWIR2"
  else:
    return "Desconocida"

# Parte C: procesar todos los archivos

fechas_unicas = set()

for archivo in archivos:
  datos = procesar_nombre_landsat(archivo)
  nombre_banda = clasificar_banda(datos["banda"])

  fechas_unicas.add(datos["fecha"])

  #Imprimir informacion

  print(f"Archivo: {archivo}")
  print(f"Sensor: {datos['sensor']}")
  print(f"Fecha: {datos['fecha']}")
  print(f"Path/Row: {datos['path']}/{datos['row']}")
  print(f"Banda: {datos['banda']} ({nombre_banda})")
  print("-" * 72)
print(f"Análisis completado. Total de fechas únicas procesadas: {len(fechas_unicas)}")

Archivo: LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF
Sensor: LC08
Fecha: 2024-06-15
Path/Row: 008/067
Banda: B4 (Rojo)
------------------------------------------------------------------------
Archivo: LC08_L2SP_008067_20240615_02_T1_SR_B5.TIF
Sensor: LC08
Fecha: 2024-06-15
Path/Row: 008/067
Banda: B5 (Infrarrojo cercano (NIR))
------------------------------------------------------------------------
Archivo: LC08_L2SP_008067_20240701_02_T1_SR_B3.TIF
Sensor: LC08
Fecha: 2024-07-01
Path/Row: 008/067
Banda: B3 (Verde)
------------------------------------------------------------------------
Archivo: LC08_L2SP_009067_20240615_02_T1_SR_B4.TIF
Sensor: LC08
Fecha: 2024-06-15
Path/Row: 009/067
Banda: B4 (Rojo)
------------------------------------------------------------------------
Archivo: LC09_L2SP_008067_20240708_02_T1_SR_B6.TIF
Sensor: LC09
Fecha: 2024-07-08
Path/Row: 008/067
Banda: B6 (SWIR1)
------------------------------------------------------------------------
Archivo: LC08_L2SP_008067_20

---
## Problema 2: Inventario Forestal en Madre de Dios (10 puntos)

El **SERFOR** realiza inventarios forestales en Madre de Dios. Los datos incluyen valores faltantes (`-999`) y mediciones con posibles errores.

**Parte A (3 pts):** Funcion `calcular_area_basal(dap_cm)`:
- Devuelve AB en m2: $AB = \pi 	\times  (DAP/200)^2$
- Devuelve `None` si DAP <= 0 o == -999

**Parte B (3 pts):** Funcion `clasificar_arbol(dap_cm, altura_m)` que devuelva:
- `clase`: "Brinzal" (<10cm), "Latizal" (10-25cm), "Fustal menor" (25-50cm), "Fustal mayor" (>=50cm)
- `alerta`: True si DAP > 200cm, altura > 60m, o altura < 1m con DAP > 10cm

**Parte C (4 pts):** Procesa los datos:
1. Para cada árbol, calcule el área basal y clasifíquelo.
2. Omita los árboles con datos faltantes (valores -999).
3. Imprima una advertencia para los árboles marcados.
4. Calcule e imprima las estadísticas descriptivas:
- Número total de árboles válidos
- Área basal total (suma de todos los árboles válidos)
- Cantidad de árboles en cada clase de tamaño
- Número de registros marcados

In [15]:
import math

# Inventario forestal - Madre de Dios, Peru (datos SERFOR)
# Formato: [id, especie, dap_cm, altura_m]
datos_arboles = [
    [1,  "Swietenia macrophylla",      35.4, 22.1],   # Caoba
    [2,  "Cedrela odorata",            28.2, 18.5],   # Cedro
    [3,  "Cedrelinga cateniformis",   -999,  25.0],   # Tornillo - DAP faltante
    [4,  "Virola surinamensis",        18.7, 12.3],   # Cumala
    [5,  "Dipteryx micrantha",         52.1, 24.8],   # Shihuahuaco
    [6,  "Calycophyllum spruceanum",    8.5,  6.2],   # Capirona
    [7,  "Terminalia oblonga",         45.0, 85.0],   # Yacushapana - altura sospechosa
    [8,  "Cedrelinga cateniformis",    62.3, 28.4],   # Tornillo
    [9,  "Swietenia macrophylla",      41.2, -999],   # Caoba - altura faltante
    [10, "Hura crepitans",             22.5,  0.5],   # Catahua - sospechoso
    [11, "Schizolobium parahybum",      5.2,  3.1],   # Pino chuncho
    [12, "Guazuma crinita",            38.9, 21.7],   # Bolaina
]

# Parte A: Escribe la función calcular_area_basal aqui:

def calcular_area_basal(dap_cm):
  if dap_cm <= 0 or dap_cm == -999:
    return None
  return math.pi * (dap_cm / 200) ** 2


# Parte B: Escribe la función clasificar_arbol aquí:

def clasificar_arbol(dap_cm, altura_m):
  if dap_cm  < 10:
    clase = "Brinzal"
  elif 10 <= dap_cm < 25:
    clase = "Latizal"
  elif 25 <= dap_cm < 50:
    clase = "Fustal menor"
  else:
    clase = "Fustal mayor"

  alerta = False

  if dap_cm > 200 or altura_m > 60 or (altura_m < 1 and dap_cm > 10):
    alerta = True

  return clase, alerta

# Parte C: Procesar los datos e imprimir los resultados

# Inicializar contadores y acumuladores
arboles_validos = 0
area_basal_total = 0.0
total_alertas = 0
conteo_clases = {
    "Brinzal": 0,
    "Latizal": 0,
    "Fustal menor": 0,
    "Fustal mayor": 0
}

for registro in datos_arboles:
    id_arbol, especie, dap_cm, altura_m = registro

    # omitir registros que tengan valores de datos faltantes (-999)
    if dap_cm == -999 or altura_m == -999:
        continue

    # área basal y clasificar árbol
    ab = calcular_area_basal(dap_cm)
    clase, alerta = clasificar_arbol(dap_cm, altura_m)

    # arboles válidos
    arboles_validos += 1
    area_basal_total += ab
    conteo_clases[clase] += 1

    # registro con alertas
    if alerta:
        total_alertas += 1
        print(f"[ALERTA] Árbol ID {id_arbol:02d} ({especie}): "
              f"DAP = {dap_cm} cm | Altura = {altura_m} m. (Clase: {clase})")

print("-" * 72)
print("                          ESTADÍSTICAS FINALES")
print("-" * 72)
print(f"Número total de árboles válidos: {arboles_validos}")
print(f"Área basal total acumulada:{area_basal_total:.4f} ")
print("-" * 72)
print("Distribución por clases de tamaño:")
for cl, cantidad in conteo_clases.items():
    print(f"{cl:<15}: {cantidad} árboles")
print("-" * 72)
print(f"Total de registros marcados con alerta de datos: {total_alertas}")


[ALERTA] Árbol ID 07 (Terminalia oblonga): DAP = 45.0 cm | Altura = 85.0 m. (Clase: Fustal menor)
[ALERTA] Árbol ID 10 (Hura crepitans): DAP = 22.5 cm | Altura = 0.5 m. (Clase: Latizal)
------------------------------------------------------------------------
                          ESTADÍSTICAS FINALES
------------------------------------------------------------------------
Número total de árboles válidos: 10
Área basal total acumulada:1.0318 
------------------------------------------------------------------------
Distribución por clases de tamaño:
Brinzal        : 2 árboles
Latizal        : 2 árboles
Fustal menor   : 4 árboles
Fustal mayor   : 2 árboles
------------------------------------------------------------------------
Total de registros marcados con alerta de datos: 2


---
## Lista de verificacion
- [ ] Todas las celdas corren sin errores
- [ ] Ambos problemas estan completos
- [ ] Salidas visibles en todas las celdas
- [ ] Nombre incluido